In [1]:
import sqlite3
import sqlite_vec
import hashlib
import json
from typing import List
import numpy as np

In [3]:
# Step 1 - Connection SQLite
conn = sqlite3.connect("demo.db")
conn.enable_load_extension(True)
sqlite_vec.load(conn)
cur = conn.cursor()

In [4]:
# Create Table
cur.execute("""
CREATE TABLE IF NOT EXISTS docs (
    id INTEGER PRIMARY KEY,
    content TEXT NOT NULL
);
""")

cur.execute("""select count(*) from docs""")
doc = [
    (1, "This is a sample document about machine learning."),
    (2, "This document discusses the applications of artificial intelligence."),
    (3, "This is a tutorial on how to use SQLite with Python."),
    (4, "This document is about natural language processing techniques."),
]
doc_count = cur.fetchone()[0]
if doc_count == 0:
    cur.executemany("INSERT INTO docs (id, content) VALUES (?, ?)", doc)



In [5]:
print(doc_count)
cur.execute("SELECT * FROM docs")
for row in cur.fetchall():
    print(row)

4
(1, 'This is a sample document about machine learning.')
(2, 'This document discusses the applications of artificial intelligence.')
(3, 'This is a tutorial on how to use SQLite with Python.')
(4, 'This document is about natural language processing techniques.')


In [6]:
def embed_text(text: str, dim: int = 8) -> List[float]:
    """
     function จำลองการสร้าง Embedding แบบง่ายๆ เพื่อการสาธิต
    ใช้ SHA256 ของข้อความเพื่อสร้างเวกเตอร์ที่ได้ผลลัพธ์เหมือนเดิมทุกครั้ง
    """
    h = hashlib.sha256(text.encode("utf-8")).digest()
    vec = []
    for i in range(dim):
        b1 = h[(i * 2) % len(h)]
        b2 = h[(i * 2 + 1) % len(h)]
        val = (b1 << 8) | b2
        f = (val / 65535.0) * 2.0 - 1.0  # ทำให้ค่าอยู่ในช่วง [-1, 1]
        vec.append(f)
    return vec

In [7]:
cur.execute("""CREATE VIRTUAL TABLE IF NOT EXISTS vec_docs USING vec0(
    embedding FLOAT[8] 
);""")

cur.execute("SELECT COUNT(*) FROM vec_docs;")
vec_count = cur.fetchone()[0]
if vec_count == 0:
    rows = []
    for doc_id, content in doc:
        emb = embed_text(content , dim=8)
        rows.append((doc_id, json.dumps(emb)))
    cur.executemany("INSERT INTO vec_docs (rowid, embedding) VALUES (?, ?)", rows)
    conn.commit()

In [ ]:
# print
cur = conn.execute("SELECT rowid, embedding FROM vec_docs LIMIT 5;")
for row in cur.fetchall():
    rowid, blob = row
    # แปลง blob กลับเป็นเวกเตอร์ float32 เพราะว่าก่อนหน้านี้เป็นการบันทึกแบบ binary เลยต้องแปลงกลับ
    vec = np.frombuffer(blob, dtype=np.float32)
    print(f"RowID: {rowid}, vec={vec[:8]}")

RowID: 1, vec=[-0.97018385 -0.61135274  0.27956054 -0.3926299   0.2773022   0.8769512
 -0.7202716   0.35106432]
RowID: 2, vec=[-0.98031586  0.55071336 -0.13786526 -0.10337988 -0.31795225 -0.3512169
 -0.23949035 -0.11006332]
RowID: 3, vec=[-0.79693294  0.6623789  -0.7719997  -0.20762952  0.7041886   0.07356375
 -0.18318456  0.10402075]
RowID: 4, vec=[ 0.56557566 -0.6204471  -0.3129778   0.04762341 -0.9982605   0.23183031
 -0.40956742  0.16993973]


In [13]:
query = "This document discusses"
query_vec = embed_text(query, dim=8)
query_vec_json = json.dumps(query_vec)
res = cur.execute("""SELECT rowid, distance FROM vec_docs where embedding MATCH ? ORDER BY distance limit 2""", (query_vec_json,)).fetchall()

for rowid, distance in res:
    print(f"- RowId={rowid}, distance={float(distance):.12f}")


- RowId=2, distance=1.744433283806
- RowId=3, distance=1.872495770454
